<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Middleware
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Was ist Middleware?
---

**Middleware** ist eine *Zwischenschicht*, die sich zwischen einen **Agenten** und die **Ausführung seiner Tools** schaltet. Sie ermöglicht es, Kontrollmechanismen einzufügen, ohne den Agenten oder die Tools zu verändern.

**Zentrale Aufgaben von Middleware:**

* **Vorverarbeitung (Preprocessing):** Bereinigung von Eingabedaten, Strukturierung von Abfragen oder Ergänzung durch Kontext
* **Kontrolle & Sicherheit:** Implementierung von Content-Filtern, PII-Anonymisierung (Personally Identifiable Information, personenbezogene Daten) und Validierung von Eingabeparametern
* **Nachbearbeitung (Postprocessing):** Formatierung der Rohdaten, Kürzung von Texten oder Anreicherung der Antwort mit Metadaten
* **Instrumentierung & Monitoring:** Systematisches Logging, Performance-Tracking sowie Debugging des gesamten Workflows

Das Konzept ist vergleichbar mit Middleware in Webframeworks (z.B. Express.js, Django): Anfragen durchlaufen eine Kette von Prüfungen, bevor sie ausgeführt werden.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Middelware-Prozess</font> </br></p>

diagram = """
flowchart TD

    A[User Input] --> B[before_model<br/>Was geht rein?]
    B --> C[MODEL<br/>LLM denkt nach]
    C --> D[after_model<br/>Was kam raus?]
    D --> E{Tool-Aufruf?}

    E -->|Nein| F[Fertig]
    E -->|Ja| G[wrap_tool_call<br/>Welches Tool? Mit welchen Args?]

    G --> B
"""
mermaid(diagram, width=600)

<p><font color='black' size="5">
Zwei Arten von Middleware
</font></p>

LangChain Version 1.0+ bietet zwei Mechanismen:

**1. Decorator-Hooks** (low-level, eigene Logik):


Ein Decorator-Hook erweitert das Verhalten einer Funktion automatisch an definierten Einhängepunkten (z. B. vor, nach oder während der Ausführung).    

| Hook | Wann? | Typische Aufgabe |
|------|-------|-----------------|
| `@before_model` | **Vor** dem LLM-Aufruf | Logging, Input-Validierung, Kontext prüfen |
| `@after_model` | **Nach** dem LLM-Aufruf | Output-Logging, Entscheidung prüfen (Tool vs. Antwort) |
| `@wrap_tool_call` | **Um** die Tool-Ausführung | Tool-Logging, Ergebnisse überwachen, Timing |



**2. Prebuilt Middleware** (high-level, fertige Bausteine):      
Prebuilt Middleware ist fertige Middleware, die standardisierte Funktionen wie Logging, Authentifizierung oder Validierung in einen Workflow integriert.

| Middleware | Zweck |
|-----------|-------|
| `HumanInTheLoopMiddleware` | Hält die Ausführung an, bevor Tools ausgeführt werden, um manuelle Freigabe einzuholen |
| `SummarizationMiddleware` | Kürzt/fasst den Nachrichtenverlauf zusammen, wenn Kontextgrenzen erreicht werden |
| `ToolRetryMiddleware` | Wiederholt fehlgeschlagene Tool-Aufrufe automatisch mit Backoff-Strategie |
| `ModelRetryMiddleware` | Wiederholt fehlgeschlagene Modellaufrufe automatisch mit Backoff |
| `ModelCallLimitMiddleware` | Begrenzt die Anzahl der LLM-Aufrufe pro Session |
| `ToolCallLimitMiddleware` | Begrenzt die Anzahl der Tool-Aufrufe (global oder pro Tool) |
| `ModelFallbackMiddleware` | Wechselt auf Backup-Modelle bei Ausfall des Primärmodells |
| `PIIMiddleware` | Erkennt und behandelt personenbezogene Daten (E-Mail, Telefon, Kreditkarten) |
| `ContextEditingMiddleware` | Entfernt/trimmt ältere Tool-Verwendungen, um die Historie schlank zu halten |
| `LLMToolEmulator` | Simuliert Tool-Ausführungen mit einem LLM für Tests ohne Nebenwirkungen |
| `LLMToolSelectorMiddleware` | Nutzt ein LLM um relevante Tools vor dem Hauptmodell-Aufruf zu filtern |
| `TodoListMiddleware` | Aufgabenplanung und Fortschrittsverfolgung für Agenten |
| `ShellToolMiddleware` | Persistente Shell-Session als Tool für Kommandoausführung |
| `FilesystemFileSearchMiddleware` | Glob/Grep-Suchtools über das Dateisystem |

Alle importierbar aus: `from langchain.agents.middleware import <Klasse>`

# 2 | Setup: Modell und Tools
---

Bevor wir Middleware einsetzen können, Definition eines LLM und einfache Tools, die der Agent verwenden kann.

In [ ]:
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    ModelRetryMiddleware,
    SummarizationMiddleware,
    ToolRetryMiddleware,
    after_model,
    before_model,
    wrap_tool_call,
)
from langchain.chat_models import init_chat_model
from langchain.tools.tool_node import ToolCallRequest
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

# LLM initialisieren
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

In [ ]:
# Tools definieren
@tool
def rechner(ausdruck: str) -> str:
    """Führt eine mathematische Berechnung durch.

    Args:
        ausdruck: Mathematischer Ausdruck, z.B. '42 * 17'

    Returns:
        Ergebnis der Berechnung
    """
    try:
        if any(x in ausdruck for x in ['import', 'exec', '__']):
            return "Unsichere Operation"
        result = eval(ausdruck)
        return f"{ausdruck} = {result}"
    except Exception:
        return "Berechnungsfehler"


@tool
def datei_lesen(dateiname: str) -> str:
    """Liest den Inhalt einer Datei.

    Args:
        dateiname: Name der zu lesenden Datei

    Returns:
        Inhalt der Datei als Text
    """
    return f"Inhalt von '{dateiname}': Dies ist ein Beispieltext."


@tool
def datei_schreiben(dateiname: str, inhalt: str) -> str:
    """Erstellt oder überschreibt eine Datei mit dem angegebenen Inhalt.

    Args:
        dateiname: Name der zu schreibenden Datei
        inhalt: Text-Inhalt der Datei

    Returns:
        Bestätigung der Schreiboperation
    """
    return f"Datei '{dateiname}' erfolgreich geschrieben."


tools = [rechner, datei_lesen, datei_schreiben]
print(f"Tools: {[t.name for t in tools]}")

# 3 | before_model / after_model / wrap_tool_call
---

Die drei Decorator-Hooks bilden das Kernstück der Middleware in LangChain 1.0+. Sie erlauben es, den gesamten Agent-Zyklus zu beobachten und zu steuern:

- `@before_model` – wird **vor** jedem LLM-Aufruf ausgeführt. Parameter: `state` (AgentState) und `runtime`.
- `@after_model` – wird **nach** jedem LLM-Aufruf ausgeführt. Parameter: `state` (AgentState) und `runtime`.
- `@wrap_tool_call` – wird **um** jeden Tool-Aufruf gewickelt. Parameter: `request` (ToolCallRequest) und `handler`.

Jeder Hook gibt `None` zurück (passiert den State unverändert durch) oder kann den State modifizieren.

<p><font color='black' size="5">
Middleware definieren
</font></p>

In [ ]:
@before_model
def log_before(state: AgentState, runtime):
    """Wird VOR jedem LLM-Aufruf ausgeführt."""
    print(f"\n🧠 Model wird aufgerufen mit {len(state['messages'])} Nachrichten")
    for msg in state["messages"][-2:]:
        role = msg.type if hasattr(msg, "type") else "unknown"
        content = str(msg.content)[:80]
        print(f"   [{role}]: {content}")
    return None


@after_model
def log_after(state: AgentState, runtime):
    """Wird NACH jedem LLM-Aufruf ausgeführt."""
    msg = state["messages"][-1]
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"⚡ Tool-Aufruf: {[tc['name'] for tc in msg.tool_calls]}")
    else:
        print(f"💬 Antwort generiert")
    return None


@wrap_tool_call
def log_tool(request: ToolCallRequest, handler):
    """Wird UM jeden Tool-Aufruf gewickelt."""
    print(f"🔧 Führe aus: {request.tool_call['name']}({request.tool_call['args']})")
    result = handler(request)
    print(f"✅ Ergebnis: {str(result.content)[:100]}...")
    return result

<p><font color='black' size="5">
Agent mit Middleware erstellen und testen
</font></p>

In [ ]:
# Agent mit allen drei Middleware-Hooks erstellen
agent = create_agent(
    model=llm,
    tools=[rechner, datei_lesen],
    system_prompt="Du bist ein hilfreicher Assistent. Antworte auf Deutsch.",
    middleware=[log_before, log_after, log_tool]
)

print("✅ Agent mit before_model / after_model / wrap_tool_call erstellt")

In [ ]:
# Test: Anfrage die ein Tool verwendet
result = agent.invoke({
    "messages": [{"role": "user", "content": "Was ist 2847 * 1923?"}]
})

print()
mprint("### 📣 Antwort")
mprint("---")
mprint(result["messages"][-1].content)

In [ ]:
# Test: Anfrage ohne Tool (direkte Antwort)
result = agent.invoke({
    "messages": [{"role": "user", "content": "Was ist Middleware?"}]
})

print()
mprint("### 📣 Antwort")
mprint("---")
mprint(result["messages"][-1].content)

<p><font color='darkblue' size="4">
ℹ️ <b>Was passiert hier?</b>
</font></p>

In der Konsole sind die Middleware-Logs sichtbar:

1. **`🧠 Model wird aufgerufen`** – `before_model` zeigt die eingehenden Nachrichten
2. **`⚡ Tool-Aufruf`** – `after_model` erkennt, dass das LLM ein Tool aufrufen möchte
3. **`🔧 Führe aus`** – `wrap_tool_call` loggt den Tool-Aufruf und das Ergebnis
4. **`🧠 Model wird aufgerufen`** – Der Agent macht eine zweite Runde mit dem Tool-Ergebnis
5. **`💬 Antwort generiert`** – `after_model` erkennt, dass jetzt eine finale Antwort kommt

# 4 | HumanInTheLoopMiddleware
---

Nicht jedes Tool sollte ohne Rückfrage ausgeführt werden. Besonders bei **schreibenden Operationen** ist eine menschliche Bestätigung sinnvoll. Die `HumanInTheLoopMiddleware` unterbricht den Agenten vor der Ausführung bestimmter Tools und fragt den Nutzer um Erlaubnis.

| Szenario | Beispiel-Tool | Warum Human-in-the-Loop? |
|----------|---------------|--------------------------|
| Dateien ändern | `datei_schreiben`, `datei_loeschen` | Datenverlust vermeiden |
| Externe Kommunikation | `send_email`, `post_message` | Ungewollte Nachrichten verhindern |
| Finanztransaktionen | `transfer_money`, `place_order` | Fehlerhafte Buchungen vermeiden |
| Systemänderungen | `deploy`, `restart_service` | Ausfälle vermeiden |

**Prinzip:** Der Agent darf *lesen* und *denken* ohne Einschränkung – aber *handeln* nur mit Zustimmung.

In [ ]:
#@markdown   <p><font size="4" color='green'>  🧜‍♀️ Human-in-the-Loop Ablauf</font> </br></p>

diagram = """
flowchart TD
    A[User Input] --> B[Agent / LLM]
    B --> C{Tool-Aufruf?}

    C -->|Nein| D[Direkte Antwort]
    C -->|Ja| E{Tool in<br/>Sperrliste?}

    E -->|Nein| F[Tool wird<br/>automatisch ausgeführt]
    E -->|Ja| G[⏸️ PAUSE<br/>Nutzer wird gefragt]

    G --> H{Nutzer<br/>erlaubt?}
    H -->|✅ Ja ... | F
    H -->|❌  Nein ... | I[Tool-Aufruf<br/>abgelehnt]

    F --> J[Ergebnis → Agent]
    I --> J
    J --> B

    style G fill:#fff3cd,stroke:#ffc107,stroke-width:2px
    style H fill:#fff3cd,stroke:#ffc107,stroke-width:2px
    style F fill:#d4edda,stroke:#28a745
    style I fill:#f8d7da,stroke:#dc3545
"""
mermaid(diagram, width=500)

<p><font color='black' size="5">
Agent mit HumanInTheLoopMiddleware erstellen
</font></p>

Für die `HumanInTheLoopMiddleware` wird ein **Checkpointer** benötigt (`MemorySaver`), damit der Agent nach der Unterbrechung an derselben Stelle fortfahren kann. Der Parameter `interrupt_on` definiert, welche Tools eine Genehmigung erfordern.

In [ ]:
# HITL-Middleware: datei_schreiben erfordert Bestätigung
hitl = HumanInTheLoopMiddleware(interrupt_on={"datei_schreiben": True})

# Agent MIT Checkpointer (nötig für Interrupt/Resume)
agent_hitl = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Du bist ein hilfreicher Datei-Assistent. Antworte auf Deutsch.",
    middleware=[hitl],
    checkpointer=MemorySaver()
)

config = {"configurable": {"thread_id": "demo-hitl"}}
print("✅ Agent mit HumanInTheLoopMiddleware erstellt")

In [ ]:
# Test: Lesen läuft ohne Unterbrechung
result = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": "Lies bitte die Datei bericht.txt"}]},
    config={"configurable": {"thread_id": "demo-lesen"}}
)

mprint("### 📣 Antwort (Lesen – ohne Genehmigung):")
mprint("---")
mprint(result["messages"][-1].content)

In [ ]:
# Test: Schreiben wird unterbrochen → Nutzer entscheidet
result = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": "Schreibe 'Hallo Welt' in die Datei test.txt"}]},
    config=config
)
print("⏸️ Agent unterbrochen – wartet auf Genehmigung\n")

# Nutzer entscheidet interaktiv
antwort = input("🔧 Tool 'datei_schreiben' ausführen? (ja/nein): ")

if antwort.strip().lower() in ("ja", "j", "yes", "y"):
    result = agent_hitl.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config
    )
    mprint(f"\n✅ {result['messages'][-1].content}")
else:
    result = agent_hitl.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}),
        config=config
    )
    print(f"\n❌ Tool abgelehnt – Agent antwortet ohne Tool:")
    print(result["messages"][-1].content)

<p><font color='darkblue' size="4">
ℹ️ <b>Wie funktioniert der Interrupt?</b>
</font></p>

1. Der Agent plant den Aufruf von `datei_schreiben`
2. Die `HumanInTheLoopMiddleware` erkennt das Tool in der `interrupt_on`-Liste
3. Der Agent wird **pausiert** – der Zustand wird im `MemorySaver` gespeichert
4. Der Nutzer gibt seine Entscheidung ein (`approve` oder `reject`)
5. `Command(resume=...)` setzt den Agenten mit der Entscheidung fort

**Wichtig:** Ohne `checkpointer=MemorySaver()` kann der Agent nach dem Interrupt nicht fortgesetzt werden!

# 5 | SummarizationMiddleware
---

Die `SummarizationMiddleware` löst ein zentrales Problem langer Konversationen: **Token-Überlauf**. Wenn eine Session zu lang wird, fasst sie den bisherigen Verlauf automatisch zusammen und hält dabei die neuesten Nachrichten im Detail.

**Parameter:**

| Parameter | Typ | Beschreibung |
|-----------|-----|-------------|
| `model` | `str` oder `BaseChatModel` | Modell für die Zusammenfassung |
| `trigger` | `ContextSize` | Token-/Nachrichtengrenze, ab der zusammengefasst wird |
| `keep` | `ContextSize` | Wie viel Kontext erhalten bleiben soll |
| `token_counter` | `callable` | Optionale Funktion zum Token-Zählen |
| `summary_prompt` | `str` | Optionales Template mit `{messages}` Platzhalter |

In [ ]:
# Agent mit automatischer Kontext-Zusammenfassung
agent_summary = create_agent(
    model=llm,
    tools=[rechner, datei_lesen],
    system_prompt="Du bist ein hilfreicher Assistent. Antworte auf Deutsch.",
    middleware=[SummarizationMiddleware(model=llm)]
)

print("✅ Agent mit SummarizationMiddleware erstellt")

**Alternative: OpenAI Server-Side Compaction** *(neu: langchain-openai 1.1.10, Feb 2026)*

Für OpenAI-Modelle gibt es seit Februar 2026 eine einfachere Alternative zur `SummarizationMiddleware`:
OpenAI komprimiert die Konversationshistorie **serverseitig** – kein extra Middleware-Layer nötig.

| Ansatz | Wie | Provider | Wann verwenden? |
|--------|-----|----------|-----------------|
| `SummarizationMiddleware` | Client: LLM fasst lokal zusammen | Alle Provider | Provider-unabhängig, mehr Kontrolle |
| `context_management` (neu) | Server: OpenAI komprimiert | Nur OpenAI | Einfachste Lösung für OpenAI-Apps |

In [ ]:
# Alternative zu SummarizationMiddleware für OpenAI-Modelle (langchain-openai 1.1.10)
llm_mit_compaction = init_chat_model(
    "openai:gpt-4o-mini",
    context_management=[{"type": "compaction", "compact_threshold": 10_000}]
    # Komprimiert automatisch, wenn ~10.000 Tokens überschritten werden
    # Kein extra Middleware-Layer nötig – alles passiert auf OpenAI-Seite
)

agent_openai = create_agent(
    model=llm_mit_compaction,
    tools=[rechner, datei_lesen],
    system_prompt="Du bist ein hilfreicher Assistent. Antworte auf Deutsch.",
    # Keine SummarizationMiddleware nötig bei OpenAI-Modellen!
)

print("✅ Agent mit OpenAI Server-Side Compaction erstellt")
print("→ compact_threshold=10000: Komprimierung ab ~10.000 Token")
print("→ Alternative zu SummarizationMiddleware(model=llm)")

# 6 | ToolRetry/ModelRetryMiddleware
---

Beide Retry-Middleware machen Agenten **robust gegen transiente Fehler**. Sie implementieren automatische Retries mit **Exponential Backoff**:

- `ToolRetryMiddleware` – wiederholt fehlgeschlagene **Tool-Aufrufe**
- `ModelRetryMiddleware` – wiederholt fehlgeschlagene **LLM-Aufrufe** (Rate Limits, Timeouts)

**Gemeinsame Parameter:**

| Parameter | Typ | Beschreibung |
|-----------|-----|-------------|
| `max_retries` | `int` | Maximale Anzahl Wiederholungen (Default: 2) |
| `backoff_factor` | `float` | Faktor für Exponential Backoff (z.B. 2.0 → 2s, 4s, 8s) |
| `initial_delay` | `float` | Anfängliche Wartezeit in Sekunden (Default: 1.0) |
| `max_delay` | `float` | Maximale Wartezeit (Default: 60s) |
| `jitter` | `bool` | ±25% zufällige Variation zur Lastvermeidung |
| `retry_on` | `tuple` | Exception-Typen, auf die reagiert wird |
| `on_failure` | `str` | Verhalten bei Erschöpfung: `'continue'`, `'error'`, `'return_message'` |

In [ ]:
# Agent mit Retry-Logik für Tools und Modell
agent_retry = create_agent(
    model=llm,
    tools=[rechner, datei_lesen],
    system_prompt="Du bist ein hilfreicher Assistent. Antworte auf Deutsch.",
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            jitter=True
        ),
        ModelRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            jitter=True
        )
    ]
)

print("✅ Agent mit ToolRetryMiddleware + ModelRetryMiddleware erstellt")

# 7 | Middleware kombinieren
---

In [ ]:
# Production-Ready Kombination: Logging + Retry + HITL + Summarization
agent_production = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Du bist ein sicherer und zuverlässiger Datei-Assistent. Antworte auf Deutsch.",
    middleware=[
        log_before,                                    # 1. Logging: Was geht rein?
        log_after,                                     # 2. Logging: Was kam raus?
        log_tool,                                      # 3. Logging: Welches Tool?
        ModelRetryMiddleware(max_retries=3, jitter=True),  # 4. LLM-Retry bei Fehlern
        ToolRetryMiddleware(max_retries=2, jitter=True),   # 5. Tool-Retry bei Fehlern
        hitl,                                          # 6. Genehmigung für Schreiboperationen
        SummarizationMiddleware(model=llm),            # 7. Kontext-Zusammenfassung
    ],
    checkpointer=MemorySaver()
)

print("✅ Production-Ready Agent erstellt")
print(f"   Middleware-Stack: 7 Schichten (Logging + Retry + HITL + Summarization)")

In [ ]:
# Test: Berechnung (wird geloggt, keine Genehmigung nötig)
result = agent_production.invoke(
    {"messages": [{"role": "user", "content": "Berechne 123 + 456"}]},
    config={"configurable": {"thread_id": "kombi-rechner"}}
)

print()
mprint("### 📣 Antwort:")
mprint("---")
mprint(result["messages"][-1].content)


**Best Practices:**

| Empfehlung | Beschreibung |
|------------|-------------|
| **Logging zuerst** | `@before_model` + `@after_model` + `@wrap_tool_call` für Transparenz |
| **HITL für Schreiboperationen** | Sensible Tools mit `interrupt_on` absichern |
| **Retry für Production** | `ModelRetryMiddleware` + `ToolRetryMiddleware` für Stabilität |
| **Summarization für lange Sessions** | `SummarizationMiddleware` verhindert Token-Overflow |
| **Checkpointer nicht vergessen** | `HumanInTheLoopMiddleware` benötigt `checkpointer=MemorySaver()` |

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> Wichtig ist nur: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**

Baue einen Agenten mit zwei Tools und einer einfachen Logging-Middleware, die jeden Tool-Aufruf protokolliert.

**✅ Erledigt wenn:** Das Logging zeigt für jeden Tool-Aufruf mindestens Zeitstempel und Tool-Name in der Ausgabe.

In [ ]:
# Grundlagen: Agent mit 2 Tools + Logging-Middleware
# Startpunkt: Agent-Beispiel aus Kapitel 2/3 als Vorlage

# 1. Zwei Tools definieren (z. B. Taschenrechner + Textverarbeitung)
# 2. Logging-Middleware: before_tool loggt Tool-Name + Zeitstempel
# 3. Agent aufrufen und Ausgabe prüfen
# ...

**Aufbau**

Integriere alle drei Middleware-Hooks (before_tool, after_tool, on_error) und füge Human-in-the-Loop für mindestens eine sensible Aktion hinzu.

**✅ Erledigt wenn:** Der Agent stoppt vor der markierten sensiblen Aktion und wartet auf Bestätigung — ohne Bestätigung wird die Aktion nicht ausgeführt.

In [ ]:
# Aufbau: Alle 3 Middleware-Hooks + Human-in-the-Loop
# Startpunkt: Middleware-Beispiele aus Kapitel 3/4

# 1. before_tool: Anfrage loggen oder modifizieren
# 2. after_tool: Antwort loggen oder nachbearbeiten
# 3. on_error: Fehler abfangen und Fallback ausgeben
# 4. HITL: für mindestens eine Aktion manuell bestätigen
# ...

**Vertiefung**

Werte Laufzeiten aus, mache die Middleware robuster gegen Ausnahmen oder ergänze weitere Guardrails (z.B. Eingabe-Validierung).

**✅ Erledigt wenn:** Die Laufzeitmessung zeigt pro Tool-Aufruf eine Zeitangabe; eine Guardrail-Verletzung erzeugt eine klare Fehlermeldung — kein stiller Fehler.

In [ ]:
# Vertiefung: Laufzeiten + Guardrails

# Option A: Laufzeitmessung
#   after_tool: time.time() Differenz pro Aufruf messen + ausgeben

# Option B: Guardrails
#   before_tool: Input auf Schlagworte prüfen (z. B. 'lösche', 'passwort')
#   Bei Verletzung: ToolException mit klarer Meldung werfen

import time
# ...